# Task 1 — dự đoán public test (360 bài)
Notebook tải `public-test-cham-diem-phan-hoi-files.zip` từ [thư mục Drive được cung cấp](https://drive.google.com/drive/folders/1kBd4pE4JHcDtm0WFYueFq6aTEuTanUNC), giải nén, kiểm tra dữ liệu và chạy pipeline zero-shot có cấu trúc. Kết quả được lưu trên Drive.

**Trước khi chạy:** đồng bộ code mới lên branch `task-1` của GitHub. Folder Drive phải cho tài khoản Colab quyền xem; nếu `gdown` không đọc được, đặt ZIP vào MyDrive và khai báo `DRIVE_ZIP`. Không có lời gọi API trong preflight.

Public test không có nhãn nên notebook không tính QWK/MAE. `predictions.json` chỉ xuất hiện khi đủ 360 dự đoán hợp lệ. Notebook không tự nộp leaderboard.

In [7]:
%pip -q install "pyyaml>=6" "openai>=2,<3" "pydantic>=2.5" gdown


In [8]:
from google.colab import drive
from pathlib import Path
import os, sys, json, zipfile, subprocess

drive.mount('/content/drive')
PROJECT_HOME = Path('/content/drive/MyDrive/LLM/Final Project')
REPO_DIR = Path('/content/llm-final-project-task1')
REPO_URL = 'https://github.com/tiendinhquang2104/llm-final-project.git'
if not (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'task-1', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', 'task-1'], check=True)
RUNNER = REPO_DIR / 'scripts' / 'run_task1.py'
if not RUNNER.is_file() or '--predict-only' not in RUNNER.read_text(encoding='utf-8'):
    raise RuntimeError('Branch task-1 trên GitHub chưa có runner public test. Đồng bộ code mới rồi chạy lại.')
sys.path.insert(0, str(REPO_DIR))
from src.data.loader import resolve_dataset_root
print('Source commit:', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Source commit: 88606a8


In [9]:
import gdown
FOLDER_URL = 'https://drive.google.com/drive/folders/1kBd4pE4JHcDtm0WFYueFq6aTEuTanUNC'
ZIP_NAME = 'public-test-cham-diem-phan-hoi-files.zip'
DRIVE_ZIP = None  # Nếu folder không tải được qua link: Path('/content/drive/MyDrive/.../public-test-cham-diem-phan-hoi-files.zip')
DOWNLOAD_DIR = Path('/content/task1-public-download')
EXTRACT_DIR = Path('/content/task1-public-extracted')
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
archive = Path(DRIVE_ZIP) if DRIVE_ZIP else next(iter(DOWNLOAD_DIR.rglob(ZIP_NAME)), None)
if archive is None:
    gdown.download_folder(url=FOLDER_URL, output=str(DOWNLOAD_DIR), quiet=False)
    archive = next(iter(DOWNLOAD_DIR.rglob(ZIP_NAME)), None)
if archive is None or not zipfile.is_zipfile(archive):
    raise RuntimeError('Không thấy ZIP hợp lệ trong folder Drive. Kiểm tra quyền xem hoặc đặt DRIVE_ZIP tới bản sao trong MyDrive.')
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive) as zf:
    root = EXTRACT_DIR.resolve()
    if any(not (root / name).resolve().is_relative_to(root) for name in zf.namelist()):
        raise ValueError('ZIP chứa đường dẫn không an toàn')
    zf.extractall(root)
# Folder Drive tải về là một ZIP tổng hợp; bộ 360 bài có thể nằm trong public_test_release.zip bên trong.
try:
    PUBLIC_ROOT = resolve_dataset_root(EXTRACT_DIR)
except FileNotFoundError:
    inner_archives = sorted(EXTRACT_DIR.rglob('public_test_release.zip'))
    if len(inner_archives) != 1:
        found = [str(p.relative_to(EXTRACT_DIR)) for p in EXTRACT_DIR.rglob('*') if p.is_file()][:20]
        raise FileNotFoundError(f'Không tìm thấy đúng một public_test_release.zip trong ZIP ngoài. File tìm thấy: {found}')
    inner_root = (EXTRACT_DIR / '_public_release_inner').resolve()
    inner_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(inner_archives[0]) as inner_zip:
        if any(not (inner_root / name).resolve().is_relative_to(inner_root) for name in inner_zip.namelist()):
            raise ValueError('ZIP public_test_release.zip chứa đường dẫn không an toàn')
        inner_zip.extractall(inner_root)
    PUBLIC_ROOT = resolve_dataset_root(inner_root)
SAMPLE_ROOT = resolve_dataset_root(PROJECT_HOME / 'data' / 'raw')
SPLITS_DIR = PROJECT_HOME / 'data' / 'splits'
OUTPUT_ROOT = PROJECT_HOME / 'data' / 'outputs' / 'task1'
if not (SPLITS_DIR / 'train_ids.json').is_file() or not (SPLITS_DIR / 'val_ids.json').is_file():
    raise FileNotFoundError('Thiếu split 26/6 trên Drive; chạy colab/01_prepare_data.ipynb trước.')
print('Public root:', PUBLIC_ROOT, 'ZIP:', archive)


Public root: /content/task1-public-extracted/_public_release_inner/public_test_release ZIP: /content/task1-public-download/public-test-cham-diem-phan-hoi-files.zip


In [14]:
PROVIDERS = ('gemini',)  # Có thể đổi thành ('openai',) hoặc ('gemini', 'openai')
RUN_LIVE = True      # Chỉ bật sau khi preflight đạt

def run_task1(provider, *flags):
    command = [sys.executable, str(RUNNER), '--provider', provider, '--data-dir', str(SAMPLE_ROOT),
               '--splits-dir', str(SPLITS_DIR), '--output-dir', str(OUTPUT_ROOT),
               '--predict-only', '--predict-root', str(PUBLIC_ROOT),
               '--predict-experiment', 'e2_zero_shot_structured', *flags]
    subprocess.run(command, cwd=REPO_DIR, check=True)

for provider in PROVIDERS:
    run_task1(provider, '--preflight')


In [15]:
from getpass import getpass
_secret_service_available = True
def load_key(name):
    global _secret_service_available
    if os.environ.get(name):
        return
    if _secret_service_available:
        try:
            from google.colab import userdata
            value = userdata.get(name)
            if value:
                os.environ[name] = value
                return
        except Exception as exc:
            if type(exc).__name__ == 'TimeoutException':
                _secret_service_available = False
            print(f'{name}: Secrets chưa sẵn sàng ({type(exc).__name__}); dùng ô nhập ẩn.')
    value = getpass(f'{name}: ').strip()
    if not value:
        raise RuntimeError(f'Thiếu {name}')
    os.environ[name] = value

if RUN_LIVE:
    for provider in PROVIDERS:
        load_key('GEMINI_API_KEY' if provider == 'gemini' else 'OPENAI_API_KEY')


GEMINI_API_KEY: Secrets chưa sẵn sàng (TimeoutException); dùng ô nhập ẩn.


In [16]:
# Smoke: một bài mỗi đề và cả hai file .cpp rỗng; checkpoint này sẽ được dùng lại khi chạy đủ 360 bài.
from src.data.loader import load_task1
public_ds = load_task1(PUBLIC_ROOT)
first = {}
for sample in sorted(public_ds.samples, key=lambda s: s.sample_id):
    first.setdefault(sample.exam_id, sample.sample_id)
smoke_ids = sorted(set(first.values()) | {s.sample_id for s in public_ds.samples if not s.code.strip()})
print('Smoke IDs:', smoke_ids)
if RUN_LIVE:
    for provider in PROVIDERS:
        run_task1(provider, '--predict-ids', ','.join(smoke_ids))


Smoke IDs: ['Class01-Test1-S01', 'Class01-Test2-S01', 'Class02-Test1-S01', 'Class03-Final-1-S01', 'Class03-Midterm-1-S01', 'Class03-Midterm-1-S58', 'Class03-Midterm-2-S01', 'Class03-Midterm-2-S56']


In [17]:
# Chạy lại cell này sau khi đạt giới hạn lượt gọi/chi phí; bài hoàn tất được lấy từ checkpoint.
if RUN_LIVE:
    for provider in PROVIDERS:
        run_task1(provider)
for path in sorted((OUTPUT_ROOT / 'public').glob('**/progress.json')):
    progress = json.loads(path.read_text(encoding='utf-8'))
    print(progress['provider'], progress['model'], f"{progress['completed']}/{progress['expected']}",
          'file nộp:', progress['submission_path'] or '(chưa đủ bài)')


gemini gemma-4-26b-a4b-it 150/360 file nộp: (chưa đủ bài)
